In [7]:
import os
import glob
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
import sys

root = os.path.abspath('../../..')
sys.path.append(root)


def plot_dgh_traces_with_breakpoints_fast(
    path_dgh,
    path_sec_files,
    save_html=False,
    output_dir=None,
    vp_column='Vertical Position m',
    sec_column='Corrected sp Cond [µS/cm]',
    id_column='ID',
    bp1_sec_column='breakpoint_1 (sec)',
    bp2_sec_column='breakpoint_2 (sec)',
    bp1_vp_column='breakpoint_1 (vp)',
    bp2_vp_column='breakpoint_2 (vp)',
    title='Traces with breakpoints'
):
    """
    Versión optimizada para Plotly:
    - Usa Scattergl (WebGL) para mayor velocidad
    - Hover mínimo: solo x e y
    - Puede guardar a HTML si save_html=True
    - El nombre del HTML se toma del archivo CSV de breakpoints
    - Sin interpolación de breakpoints

    Parámetros
    ----------
    path_dgh : str
        Ruta al CSV con los breakpoints.
    path_sec_files : str
        Carpeta donde están los CSV de traces.
    save_html : bool
        Si True, guarda la figura en HTML.
    output_dir : str or None
        Directorio donde guardar el HTML. Obligatorio si save_html=True.
    """

    # -------------------------
    # Validaciones de guardado
    # -------------------------
    if save_html:
        if output_dir is None:
            raise ValueError("Si save_html=True, debes proporcionar output_dir.")
        os.makedirs(output_dir, exist_ok=True)

    # -------------------------
    # Leer y validar archivo DGH
    # -------------------------
    df_dgh = pd.read_csv(path_dgh)

    required_dgh_cols = [
        id_column,
        bp1_sec_column, bp2_sec_column,
        bp1_vp_column, bp2_vp_column
    ]
    missing_dgh = [c for c in required_dgh_cols if c not in df_dgh.columns]
    if missing_dgh:
        raise ValueError(f"Faltan columnas en path_dgh: {missing_dgh}")

    df_dgh = df_dgh[required_dgh_cols].copy()
    df_dgh[id_column] = df_dgh[id_column].astype(str)

    for col in [bp1_sec_column, bp2_sec_column, bp1_vp_column, bp2_vp_column]:
        df_dgh[col] = pd.to_numeric(df_dgh[col], errors='coerce')

    # -------------------------
    # Colores
    # -------------------------
    unique_ids = df_dgh[id_column].dropna().unique().tolist()
    palette = px.colors.qualitative.Dark24
    color_map = {id_val: palette[i % len(palette)] for i, id_val in enumerate(unique_ids)}

    fig = go.Figure()
    shown_legend_ids = set()

    # -------------------------
    # Iterar por cada ID
    # -------------------------
    for _, row in df_dgh.iterrows():
        id_val = str(row[id_column])
        color = color_map[id_val]

        breakpoints = [
            (row[bp1_sec_column], row[bp1_vp_column], 1),
            (row[bp2_sec_column], row[bp2_vp_column], 2),
        ]

        pattern = os.path.join(path_sec_files, f"{id_val}*.csv")
        matched_files = sorted(glob.glob(pattern))

        if not matched_files:
            print(f"[AVISO] No se encontraron archivos para ID={id_val} con patrón: {pattern}")
            continue

        for file_path in matched_files:
            try:
                df_trace = pd.read_csv(file_path, usecols=[vp_column, sec_column])
            except Exception as e:
                print(f"[AVISO] No se pudo leer {file_path}: {e}")
                continue

            df_trace[vp_column] = pd.to_numeric(df_trace[vp_column], errors='coerce')
            df_trace[sec_column] = pd.to_numeric(df_trace[sec_column], errors='coerce')
            df_trace = df_trace.dropna(subset=[vp_column, sec_column])

            if df_trace.empty:
                print(f"[AVISO] {os.path.basename(file_path)} quedó vacío después de limpiar NaNs.")
                continue

            x = df_trace[sec_column].to_numpy(dtype=np.float32)
            y = df_trace[vp_column].to_numpy(dtype=np.float32)

            showlegend_line = id_val not in shown_legend_ids

            # Trace principal con WebGL
            fig.add_trace(
                go.Scattergl(
                    x=x,
                    y=y,
                    mode='markers + lines',
                    name=id_val,
                    legendgroup=id_val,
                    showlegend=showlegend_line,
                    marker=dict(
                        color=color,
                        size=3
                    ),
                    hovertemplate='x=%{x:.2f}<br>y=%{y:.2f}<extra></extra>'
                )
            )

            shown_legend_ids.add(id_val)

            # Breakpoints
            bp_x = []
            bp_y = []

            for bp_sec, bp_vp, i_bp in breakpoints:
                if pd.isna(bp_sec) or pd.isna(bp_vp):
                    continue

                bp_x.append(bp_sec)
                bp_y.append(bp_vp)

                # Línea horizontal opcional
                fig.add_shape(
                    type='line',
                    x0=float(np.min(x)),
                    x1=float(np.max(x)),
                    y0=float(bp_vp),
                    y1=float(bp_vp),
                    line=dict(color=color, dash='dot', width=1),
                    opacity=0.35
                )

            if bp_x:
                fig.add_trace(
                    go.Scattergl(
                        x=np.array(bp_x, dtype=np.float32),
                        y=np.array(bp_y, dtype=np.float32),
                        mode='markers',
                        legendgroup=id_val,
                        showlegend=False,
                        marker=dict(
                            color=color,
                            size=10,
                            symbol='x'
                        ),
                        hovertemplate='x=%{x:.2f}<br>y=%{y:.2f}<extra></extra>'
                    )
                )

    fig.update_layout(
        title=title,
        xaxis_title=sec_column,
        yaxis_title=vp_column,
        template='plotly_white',
        legend_title='ID',
        hovermode='closest',
        height=600
    )

    fig.update_yaxes(autorange='reversed')

    # -------------------------
    # Guardar HTML
    # -------------------------
    if save_html:
        base_name = os.path.splitext(os.path.basename(path_dgh))[0]
        output_html = os.path.join(output_dir, f"{base_name}.html")

        fig.write_html(
            output_html,
            include_plotlyjs='cdn',
            full_html=True
        )
        print(f"[OK] HTML guardado en: {output_html}")

    return fig

# SEC

In [9]:
path_dgh = f'{root}/results/dgh_breakpoints/dgh_27800_raw_sat51w2p_priority.csv'
output_dir = f'{root}/results/html_breakpoints'
path_sec_files = f'{root}/data/rawdy/rawdy_sat51w2p_priority'

fig = plot_dgh_traces_with_breakpoints_fast(
    path_dgh=path_dgh,
    path_sec_files=path_sec_files,
    vp_column='Vertical Position m',
    sec_column='Corrected sp Cond [µS/cm]',
    save_html=True,
    output_dir=output_dir,
    title='SEC Priority wells'
)

fig.show()


[OK] HTML guardado en: c:\Users\Mariana\Documents\freshwater_lens/results/html_breakpoints\dgh_27800_raw_sat51w2p_priority.html


# TEMP

In [ ]:
path_dgh = f'{root}/results/dgh_breakpoints/dgh_27800_raw_sat51w2p_lrs70.csv'
output_dir = f'{root}/results/html_breakpoints'
path_sec_files = f'{root}/data/rawdy/rawdy_sat51w2p_lrs70'

fig = plot_dgh_traces_with_breakpoints_fast(
    path_dgh=path_dgh,
    path_sec_files=path_sec_files,
    vp_column='Vertical Position m',
    sec_column='Temp °C',
    save_html=True,
    output_dir=output_dir,
    title='Temperature SEC LRS70'
)

fig.show()